# BirdCLEF+ 2025 - Pretrained models, A2: Perch 2.0 embeddings + small heads (GPU)

This notebook uses a frozen Perch 2.0 model (community ONNX `Calotriton/perch-v2-onnx`) as a feature extractor and trains our own logistic-regression and small-MLP heads on top. Perch's logits are not used anywhere. Perch 2.0 was released after the 2025 competition closed, so we point this out in the report.

**Embedding:** we cut clips to 30 s and use non-overlapping 5 s windows with the 1536-d `embedding` output. A clip shorter than 5 s gives one zero-padded window, and a trailing part shorter than 5 s is dropped.

**Labels:** every window inherits its clip's labels (primary 1.0, secondary 0.5). Windows are weighted by 1 / (windows in the clip), so each clip counts once per epoch, as with v2's one random crop per clip.

**Split:** we use our team's baseline split (stratified 80/20, seed 42) on its `train_index.csv`. The split is on clips, so all windows of a clip stay in one fold.

**Validation:** we follow the baseline protocol (first 5 s of each hold-out clip, primary label only, macro ROC-AUC skipping classes with no positives), and this is what picks the head. We also report max-over-windows per clip.

In [ ]:
import os
import torch

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local CPU test only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE_TEST:
    print("SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")

In [ ]:
import subprocess, sys
from pathlib import Path

OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
ORT_VERSION = "1.31.1"

if not SMOKE_TEST:
    # torch is already imported above, which makes its CUDA libraries available to onnxruntime.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"onnxruntime-gpu=={ORT_VERSION}"], check=True)
    print("torch CUDA:", torch.version.cuda, "| cuDNN:", torch.backends.cudnn.version())
    # CPU wheels for the internet-off inference notebook, which reads this notebook's output.
    wheels = OUT / "ort_cpu_wheels"; wheels.mkdir(exist_ok=True)
    this_py = f"{sys.version_info.major}.{sys.version_info.minor}"
    for py in ["3.11", "3.12", "3.13"]:
        r = subprocess.run([sys.executable, "-m", "pip", "download", "-q", f"onnxruntime=={ORT_VERSION}",
                            "--no-deps", "--only-binary=:all:", "--python-version", py, "--implementation", "cp",
                            "--platform", "manylinux_2_27_x86_64", "--platform", "manylinux_2_28_x86_64",
                            "-d", str(wheels)], capture_output=True, text=True)
        if r.returncode != 0 and py == this_py:
            raise RuntimeError(f"No onnxruntime wheel for this Python ({py}): {r.stderr[-400:]}")
    print("CPU wheels:", sorted(p.name for p in wheels.glob("*.whl")))

In [ ]:
import ast, glob, hashlib, json, tempfile, time, urllib.request
import numpy as np
import pandas as pd
import soundfile as sf
import torch.nn as nn
from scipy.signal import resample_poly
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import onnxruntime as ort

CFG = dict(max_seconds=30, val_frac=0.2, seed=42, secondary_weight=0.5,
           embed_batch=64, epochs=5 if SMOKE_TEST else 30, batch_size=1024, lr=1e-3,
           weight_decay=1e-2, mlp_hidden=512, mlp_dropout=0.3, preprocess_version=1)
NW = int(os.environ.get("M3_WORKERS", "0" if SMOKE_TEST else "4"))   # audio-loading workers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"]); torch.cuda.manual_seed_all(CFG["seed"])

def find_dir(marker, env, fallback):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    return Path(hits[0]).parent if hits else Path(fallback)

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR", "../../data")
MELS = find_dir("train_index.csv", "M3_MELS_DIR", "../../outputs/jamie_make_mels")
print("Data:", DATA, "| M1 index:", MELS, "| device:", DEVICE, "| ORT", ort.__version__, ort.get_available_providers())

## Clips, labels and the baseline's exact split

In [ ]:
index = pd.read_csv(MELS / "train_index.csv")      # same table, in the same order, that M1 split
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

if "filename" in index.columns:
    filenames = index["filename"].astype(str).tolist()
else:
    # Only safe when M1's index is row-for-row train.csv; anything else must stop here.
    train_csv = pd.read_csv(DATA / "train.csv")
    same = len(train_csv) == len(index) and (train_csv["primary_label"].astype(str).values
                                             == index["primary_label"].astype(str).values).all()
    if not same:
        raise RuntimeError(f"train_index.csv has no filename column and does not align with train.csv; "
                           f"columns: {list(index.columns)}")
    filenames = train_csv["filename"].astype(str).tolist()
    print("train_index.csv has no filename column: aligned to train.csv row by row")
files = [DATA / "train_audio" / f for f in filenames]

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

# Copied from M1 train-v2: same call, same array of clip numbers, same stratify, same seed.
idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
assert not set(idx_tr) & set(idx_va)
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va))
pd.DataFrame({"clip": idx_va, "filename": [filenames[i] for i in idx_va]}).to_csv(OUT / "val_clips.csv", index=False)

## Extract embeddings once (reused from disk when already present)

In [ ]:
SR, WIN_SEC = 32_000, 5
WIN = SR * WIN_SEC          # 160000 samples, the fixed Perch 2.0 input
PERCH_SHA256 = "bf0c8467a924cb074663970ca4a0ab1e143602121930209657d0dff5d5cefa1f"

def to_32k_mono(y, sr):
    """Shared with the other A2 notebook, keep both copies identical."""
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = resample_poly(y, SR, sr)     # polyphase, as perch-hoplite does
    return y.astype(np.float32)

def preprocess_windows(x):
    """Perch preprocessing, per window: remove the mean, scale the peak to 0.25.
    Shared with the other A2 notebook, keep both copies identical."""
    x = x - x.mean(axis=1, keepdims=True)
    peak = np.abs(x).max(axis=1, keepdims=True)
    return (x * (0.25 / np.maximum(peak, 1e-8))).astype(np.float32)

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Shared with the other A2 notebook."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

In [ ]:
def load_clip_windows(path):
    with sf.SoundFile(str(path)) as f:
        sr = f.samplerate
        y = f.read(min(len(f), CFG["max_seconds"] * sr), dtype="float32", always_2d=True)
    y = to_32k_mono(y, sr)
    n_full = len(y) // WIN
    if n_full == 0:
        y = np.pad(y, (0, WIN - len(y)))
        n_full = 1
    return preprocess_windows(np.stack([y[i * WIN:(i + 1) * WIN] for i in range(n_full)]))

class ClipDataset(torch.utils.data.Dataset):
    def __len__(self):
        return len(files)
    def __getitem__(self, j):
        try:
            return j, load_clip_windows(files[j]), ""
        except Exception as e:             # counted and limited below
            return j, np.zeros((0, WIN), np.float32), repr(e)

def keep_items(items):
    return items

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

def get_onnx():
    if os.environ.get("M3_PERCH_ONNX"):
        return Path(os.environ["M3_PERCH_ONNX"])
    target = Path("/tmp/perch_v2.onnx")     # outside /kaggle/working, keeps 409 MB out of the output
    if not target.exists():
        url = ("https://huggingface.co/Calotriton/perch-v2-onnx/resolve/"
               "bb0f724ee99fa4e0dc3637d8bbdc6a0e26660480/perch_v2.onnx")
        t0 = time.time(); urllib.request.urlretrieve(url, target); print(f"downloaded in {time.time() - t0:.0f}s")
    got = sha256_of(target)
    if got != PERCH_SHA256:
        raise RuntimeError(f"perch_v2.onnx checksum mismatch: {got}")
    return target

GPU_PROVIDERS = [("CUDAExecutionProvider", {"device_id": 0}), "CPUExecutionProvider"]
PROVIDERS = ["CPUExecutionProvider"] if SMOKE_TEST else GPU_PROVIDERS

def check_gpu_placement(onnx_path, batch):
    """get_providers() can list CUDA while the graph really runs on CPU, so time each node."""
    so = ort.SessionOptions(); so.enable_profiling = True
    so.profile_file_prefix = str(Path(tempfile.gettempdir()) / "ort_probe")
    s = ort.InferenceSession(str(onnx_path), so, providers=PROVIDERS)
    s.run(["embedding"], {"inputs": batch}); s.run(["embedding"], {"inputs": batch})
    events = json.load(open(s.end_profiling()))
    by = {}
    for e in events:
        if e.get("cat") == "Node" and str(e.get("name", "")).endswith("_kernel_time"):
            p = e.get("args", {}).get("provider", "?")
            by[p] = by.get(p, 0) + e.get("dur", 0)
    total = sum(by.values())
    print("Node kernel time by provider (us):", by)
    if SMOKE_TEST:
        return
    share = by.get("CUDAExecutionProvider", 0) / total if total else 0.0
    print(f"CUDA share of node time: {share:.1%}")
    if share < 0.5:
        raise RuntimeError(f"Only {share:.1%} of node time is on CUDA: refusing to run on CPU")

def fingerprint():
    return dict(filenames_sha=hashlib.sha256("\n".join(filenames).encode()).hexdigest(),
                perch_sha=PERCH_SHA256, max_seconds=CFG["max_seconds"],
                preprocess_version=CFG["preprocess_version"])

def find_cache():
    want = fingerprint()
    for d in [OUT] + [Path(p).parent for p in glob.glob("/kaggle/input/**/perch_embeddings.npy", recursive=True)]:
        meta = d / "perch_cache_meta.json"
        if meta.exists() and (d / "perch_embeddings.npy").exists() and (d / "perch_windows.csv").exists():
            if json.load(open(meta)).get("fingerprint") == want:
                return d
    return None

cache = find_cache()
if cache is None:
    onnx_path = get_onnx()
    t0 = time.time()
    check_gpu_placement(onnx_path, np.zeros((CFG["embed_batch"], WIN), np.float32))
    print(f"placement probe {time.time() - t0:.0f}s")
    sess = ort.InferenceSession(str(onnx_path), providers=PROVIDERS)
    print("Active providers:", sess.get_providers())
    if not SMOKE_TEST and sess.get_providers()[0] != "CUDAExecutionProvider":
        raise RuntimeError(f"CUDAExecutionProvider is not active: {sess.get_providers()}")
    assert sess.get_inputs()[0].name == "inputs", sess.get_inputs()[0].name
    assert "embedding" in [o.name for o in sess.get_outputs()]

    loader = torch.utils.data.DataLoader(
        ClipDataset(), batch_size=8, shuffle=False, num_workers=NW, collate_fn=keep_items,
        prefetch_factor=4 if NW else None)     # numpy goes straight to ORT, so pin_memory has no effect
    BATCH = CFG["embed_batch"]
    embs, clip_ids, win_ids, failed = [], [], [], []
    pending, n_pending, n_done = [], 0, 0
    t_start = time.time(); t_gpu = 0.0

    def run_batches(x):
        global t_gpu
        for i in range(0, len(x), BATCH):
            t = time.time()
            embs.append(sess.run(["embedding"], {"inputs": x[i:i + BATCH]})[0])
            t_gpu += time.time() - t

    for step, items in enumerate(loader):
        for j, w, err in items:
            if err:
                failed.append((filenames[j], err)); continue
            clip_ids.extend([j] * len(w)); win_ids.extend(range(len(w)))
            pending.append(w); n_pending += len(w)
        if n_pending >= BATCH:
            x = np.concatenate(pending)
            n = len(x) // BATCH * BATCH
            run_batches(x[:n]); n_done += n
            pending = [x[n:]] if n < len(x) else []; n_pending = len(x) - n
        if step % 200 == 0:
            done_clips = (step + 1) * 8; el = time.time() - t_start
            print(f"{done_clips}/{len(files)} clips | {n_done} windows | {el:.0f}s | "
                  f"ETA {el / done_clips * (len(files) - done_clips) / 60:.1f} min | model {t_gpu:.0f}s", flush=True)
    if pending:
        run_batches(np.concatenate(pending))

    E = np.concatenate(embs).astype(np.float32)
    assert E.shape == (len(clip_ids), 1536), E.shape
    print(f"failed decodes: {len(failed)}", failed[:5])
    if len(failed) > max(3, 0.01 * len(files)):
        raise RuntimeError("Too many clips failed to decode")
    np.save(OUT / "perch_embeddings.tmp.npy", E)
    os.replace(OUT / "perch_embeddings.tmp.npy", OUT / "perch_embeddings.npy")
    pd.DataFrame({"clip": clip_ids, "win": win_ids}).to_csv(OUT / "perch_windows.csv", index=False)
    wall = time.time() - t_start
    json.dump(dict(fingerprint=fingerprint(), windows=len(E), clips=len(files), failed=failed, wall_s=wall,
                   model_s=t_gpu, windows_per_s=len(E) / wall), open(OUT / "perch_cache_meta.json", "w"), indent=1)
    print(f"TIMING | {len(E)} windows, {len(files)} clips | wall {wall:.0f}s | model {t_gpu:.0f}s | {len(E) / wall:.0f} windows/s")
    del sess, embs
    cache = OUT
else:
    print("Reusing cached embeddings from", cache)

E = np.load(cache / "perch_embeddings.npy")
windows = pd.read_csv(cache / "perch_windows.csv")
assert E.shape[1] == 1536 and len(E) == len(windows)
print("embeddings", E.shape)

## Heads, trained from the cache only

In [ ]:
clip_of, win_of = windows["clip"].values, windows["win"].values
in_train = np.isin(clip_of, idx_tr); in_val = np.isin(clip_of, idx_va)
assert not (in_train & in_val).any() and (in_train | in_val).all()

Y = np.zeros((len(windows), len(species)), np.float32)      # train: primary 1.0, secondary 0.5
for row, c in enumerate(clip_of):
    Y[row, sec_idx[c]] = CFG["secondary_weight"]
    Y[row, y_idx[c]] = 1.0
clip_windows = pd.Series(clip_of).map(pd.Series(clip_of).value_counts()).values
W_row = (1.0 / clip_windows).astype(np.float32)             # each clip counts once

mean, std = E[in_train].mean(axis=0), E[in_train].std(axis=0) + 1e-6
Z = (E - mean) / std

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro"), int(keep.sum())

# Validation rows, M1 protocol: first window, primary label only
rows_va = np.where(in_val & (win_of == 0))[0]
Y_va = np.zeros((len(rows_va), len(species)), np.float32)
Y_va[np.arange(len(rows_va)), y_idx[clip_of[rows_va]]] = 1.0

rows_all_va = np.where(in_val)[0]
va_groups = clip_of[rows_all_va]
order_clips = np.unique(va_groups)
Y_clip = np.zeros((len(order_clips), len(species)), np.float32)
Y_clip[np.arange(len(order_clips)), y_idx[order_clips]] = 1.0

def make_head(kind):
    d = len(species)
    if kind == "logreg":
        return nn.Sequential(nn.Linear(1536, d))
    return nn.Sequential(nn.Linear(1536, CFG["mlp_hidden"]), nn.ReLU(), nn.Dropout(CFG["mlp_dropout"]),
                         nn.Linear(CFG["mlp_hidden"], d))

Z_dev = torch.from_numpy(Z).to(DEVICE); Y_dev = torch.from_numpy(Y).to(DEVICE); W_dev = torch.from_numpy(W_row).to(DEVICE)
tr_rows = torch.from_numpy(np.where(in_train)[0]).to(DEVICE)

@torch.no_grad()
def predict(model, rows):
    model.eval(); out = []
    for i in range(0, len(rows), 4096):
        out.append(torch.sigmoid(model(Z_dev[torch.from_numpy(rows[i:i + 4096]).to(DEVICE)])).cpu().numpy())
    return np.concatenate(out)

def evaluate(model):
    first, n_cls = macro_auc(Y_va, predict(model, rows_va))
    p = pd.DataFrame(predict(model, rows_all_va)).groupby(va_groups).max().loc[order_clips].values
    return first, macro_auc(Y_clip, p)[0], n_cls

def train_head(kind):
    torch.manual_seed(CFG["seed"])
    model = make_head(kind).to(DEVICE)
    steps = int(np.ceil(len(tr_rows) / CFG["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=CFG["epochs"] * steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    best = dict(first=-1.0)
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); t0 = time.time(); total = 0.0
        perm = tr_rows[torch.randperm(len(tr_rows), device=DEVICE)]
        for s in range(steps):
            b = perm[s * CFG["batch_size"]:(s + 1) * CFG["batch_size"]]
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                bce = nn.functional.binary_cross_entropy_with_logits(model(Z_dev[b]).float(), Y_dev[b], reduction="none")
                loss = (bce.mean(1) * W_dev[b]).sum() / W_dev[b].sum()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            total += loss.item()
        first, clip_max, n_cls = evaluate(model)
        flag = ""
        if first > best["first"]:
            best = dict(first=first, clip_max=clip_max, epoch=ep, n_classes=n_cls,
                        state={k: v.cpu().clone() for k, v in model.state_dict().items()}); flag = "  <- best"
        print(f"{kind:6s} ep {ep:2d} | loss {total / steps:.4f} | val AUC first-5s {first:.4f} | clip-max {clip_max:.4f} | {time.time() - t0:.1f}s{flag}", flush=True)
    model.load_state_dict(best["state"])
    return model, best

results = {k: train_head(k) for k in ("logreg", "mlp")}
print("\nHead   | val AUC first-5s (M1 protocol) | val AUC clip-max | best epoch | classes")
for k, (_, b) in results.items():
    print(f"{k:6s} | {b['first']:.4f} | {b['clip_max']:.4f} | {b['epoch']} | {b['n_classes']}")

## Save the better head and the species order

In [ ]:
best_kind = max(results, key=lambda k: results[k][1]["first"])
model, best = results[best_kind]
linears = [m for m in model if isinstance(m, nn.Linear)]
arrays = dict(mean=mean, std=std, n_layers=len(linears))
for i, m in enumerate(linears):
    arrays[f"W{i}"] = m.weight.detach().cpu().numpy(); arrays[f"b{i}"] = m.bias.detach().cpu().numpy()
np.savez(OUT / "perch_head.npz", **arrays)
json.dump(species, open(OUT / "species.json", "w"))

head = np.load(OUT / "perch_head.npz")       # inference path must reproduce the trained head
np_p = head_predict(head, E[rows_va])
assert np.allclose(np_p, predict(model, rows_va), atol=1e-4), "numpy head differs from torch head"

summary = dict(best_head=best_kind, val_auc_first5s=best["first"], val_auc_clip_max=best["clip_max"],
               val_classes=best["n_classes"], perch_sha256=PERCH_SHA256, cfg=CFG,
               heads={k: {x: b[x] for x in ("first", "clip_max", "epoch")} for k, (_, b) in results.items()})
json.dump(summary, open(OUT / "head_cfg.json", "w"), indent=1)
print("best:", best_kind, f"val AUC {best['first']:.4f} | saved perch_head.npz, species.json, head_cfg.json")